## 8. Neural networks and deep learning

This week, we'll learn about neural nets and build a model for classifying images of clothes
8.1 Fashion classification

The dataset can be accessed here:
- Small: https://github.com/alexeygrigorev/clothing-dataset-small
- Full: https://github.com/alexeygrigorev/clothing-dataset


QUICK REFERENCE: KEY TERMS

| Term | Definition |
|---|---|
| **CNN** | Convolutional Neural Network - a deep learning architecture specialized for image data |
| **Filter / Kernel** | A small learnable weight matrix slid across the input to detect local patterns |
| **Convolution** | The operation of sliding a filter across an input and computing dot products |
| **Feature Map** | The 2D output of applying one filter to one input; shows where the pattern is detected |
| **Stride** | How many pixels the filter moves at each step (stride=1 is most common) |
| **Padding** | Adding zeros around the input border so the output size matches the input size |
| **Pooling** | Downsampling operation that reduces feature map spatial dimensions |
| **ReLU** | Activation function `max(0, x)` - introduces non-linearity |
| **Dense Layer** | Fully connected layer; each output connects to every input via a weight matrix |
| **Softmax** | Activation that turns a score vector into a valid probability distribution |
| **Sigmoid** | Activation that squashes a single score into a [0,1] probability |
| **Feature Vector / Embedding** | The 1D representation of an image produced after the conv layers |
| **Weight Sharing** | All positions in the image use the same filter weights > fewer parameters |
| **Hierarchical Features** | Deeper layers learn more complex features built from simpler ones in earlier layers |

In [3]:
import numpy as np

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img

I0000 00:00:1779293072.286888    9651 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779293072.289712    9651 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779293072.562072    9651 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779293073.981928    9651 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [4]:
path = 'data/train/t-shirt'
name = '5f0a3fa0-6a3d-4b68-b213-72766a643de7.jpg'
fullname = f'{path}/{name}'

# load_img(fullname)

When loading an image, you can specify the size. The reason we need to do this is that a neural network expects an image of a certain size, usually 299×299, 224×224, or 150×150. If the original size of our image is larger, we need to resize it to one of the mentioned formats. Resizing is quite easy; you only need to use the target_size parameter. Here’s an example:

In [5]:
img = load_img(fullname, target_size=(299, 299))

### 8.3 PRE-TRAINED CONVOLUTIONAL NEURAL NETWORKS

This time we want to take an image and an off-the-shelf neural network that was already trained by somebody, so we can use it. Now we want to use a special model called **“Xception”** from Keras which was trained on **ImageNet** (https://www.image-net.org/). You can find more pre-trained models on Keras. Before defining the model we need some imports first.

In [6]:
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.applications.xception import decode_predictions

model = Xception(
    weights="imagenet",
    input_shape=(299, 299, 3)
)

E0000 00:00:1779293075.135302    9651 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Now we want to use this model to classify the image we used before. But this time the model.predict function expects a bunch of images. So let’s create an array with possibly multiple images. In this case it is just one.

In [7]:
x = np.array(img)
X = np.array([x])

To do the prediction, we need some preprocessing before. This model expects inputs in a certain way using the preprocess_input function.

In [8]:
X = preprocess_input(X)
X[0]

array([[[ 0.4039216 ,  0.3411765 , -0.2235294 ],
        [ 0.4039216 ,  0.3411765 , -0.2235294 ],
        [ 0.41960788,  0.35686278, -0.20784312],
        ...,
        [ 0.96862745,  0.9843137 ,  0.94509804],
        [ 0.96862745,  0.9843137 ,  0.94509804],
        [ 0.96862745,  0.99215686,  0.9372549 ]],

       [[ 0.47450984,  0.4039216 , -0.12156862],
        [ 0.4666667 ,  0.39607847, -0.12941176],
        [ 0.45882356,  0.38823533, -0.15294117],
        ...,
        [ 0.96862745,  0.9764706 ,  0.9372549 ],
        [ 0.96862745,  0.9764706 ,  0.9372549 ],
        [ 0.96862745,  0.9764706 ,  0.92941177]],

       [[ 0.56078434,  0.48235297, -0.00392157],
        [ 0.5686275 ,  0.4901961 ,  0.00392163],
        [ 0.5686275 ,  0.49803925, -0.01176471],
        ...,
        [ 0.9607843 ,  0.96862745,  0.92156863],
        [ 0.9607843 ,  0.96862745,  0.92156863],
        [ 0.9607843 ,  0.96862745,  0.92156863]],

       ...,

       [[ 0.2941177 ,  0.18431377, -0.40392154],
        [ 0

In [9]:
pred = model.predict(X)
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 616ms/step


array([[3.23711865e-04, 1.57383693e-04, 2.13493026e-04, 1.52370369e-04,
        2.47626012e-04, 3.05035879e-04, 3.20591993e-04, 1.47499173e-04,
        2.03621661e-04, 1.49272295e-04, 1.95662724e-04, 2.10137208e-04,
        7.59264294e-05, 1.13972121e-04, 1.62683398e-04, 2.04638447e-04,
        1.97415793e-04, 1.44288744e-04, 1.40217366e-04, 1.73685810e-04,
        7.46690319e-04, 2.56966421e-04, 2.66808376e-04, 2.96513957e-04,
        3.73601797e-04, 2.77404120e-04, 2.16570872e-04, 2.27269964e-04,
        3.80812795e-04, 1.72165805e-04, 3.05400754e-04, 1.96431152e-04,
        3.92114831e-04, 4.78071044e-04, 2.91751057e-04, 3.25693080e-04,
        1.47395112e-04, 1.62361932e-04, 2.12710584e-04, 1.34028145e-04,
        2.40070309e-04, 6.75211253e-04, 2.54943239e-04, 1.44478487e-04,
        4.12820780e-04, 2.04408469e-04, 3.02958069e-04, 1.49339496e-04,
        1.99653543e-04, 2.27005672e-04, 2.93729157e-04, 2.27444500e-04,
        6.37644203e-04, 7.82615505e-04, 2.49557226e-04, 4.052703

Each value is the probability that this image belongs to some class. To be able to make sense from this output, we need to know what are the classes. Therefor we need another function called decode_predictions to make the prediction human readable.

In [10]:
decode_predictions(pred)

[[('n03595614', 'jersey', np.float32(0.6819633)),
  ('n02916936', 'bulletproof_vest', np.float32(0.03814009)),
  ('n04370456', 'sweatshirt', np.float32(0.034324754)),
  ('n03710637', 'maillot', np.float32(0.011354214)),
  ('n04525038', 'velvet', np.float32(0.0018453592))]]

In real this image is a t-shirt, but **ImageNet is not very good when it comes to clothes detection**. That means it doesn’t really work for our purpose here. That means we need to train a different model with the classes we need for our case. Good point here, we don’t have to retrain the model from scratch. We can reuse this model. That means we can build on top of what big companies or universities have provided and adapt to our specific use case.

## 8.4 COVOLUTIONAL NEURAL NETWORKS

Before CNNs existed, people tried to classify images using **fully connected (dense) networks** - treating every pixel as an independent input feature.

This causes two major problems:

| Problem | Why it matters |
|---|---|
| **Too many parameters** | A 224×224 RGB image has 150,528 pixels. One dense layer with 1,024 units needs ~154 million weights. |
| **No spatial awareness** | Dense layers treat pixel (0,0) and pixel (100,100) as equally related. They ignore that nearby pixels share structure. |

**CNNs solve both problems** by using small, reusable filters that slide across the image - sharing weights and explicitly exploiting local spatial structure.

> **Key intuition:** A filter that detects a horizontal edge should work the same way whether that edge appears in the top-left or the bottom-right of the image. CNNs enforce this with **weight sharing**.

#### CNN ARCHITECTURE OVERVIEW

Think of a CNN as a pipeline with two stages:

```
IMAGE
  │
  ▼
┌─────────────────────────────────┐
│   FEATURE EXTRACTION STAGE      │
│  Conv Layer > Conv Layer > ...  │
│  (learns WHAT is in the image)  │
└─────────────────────────────────┘
  │
  ▼  (FLATTENED VECTOR)
┌─────────────────────────────────┐
│   CLASSIFICATION STAGE          │
│   Dense Layer > Dense Layer ... │
│   (decides the final label)     │
└─────────────────────────────────┘
  │
  ▼
PREDICTION (e.g. "cat" - 92%)
```

Each stage has a clearly different responsibility:
- **Convolutional layers** > extract visual features (edges, textures, shapes, objects)
- **Dense layers** > combine those features to produce a class probability

## 8.4.3. CONVOLUTIONAL LAYERS & FILTERS

#### 8.4.3.1 WHAT IS A FILTER?

A **filter** (also called a *kernel*) is a small matrix of learnable weights - typically **3×3** or **5×5** pixels. Each filter is designed (through training) to detect a specific **local pattern** in the image.

```
Example of a 3×3 vertical-edge filter:

  [ -1   0   1 ]
  [ -1   0   1 ]
  [ -1   0   1 ]

> Responds strongly where pixel intensity changes sharply from left to right.
```

In early layers, filters tend to capture **simple patterns** (edges, corners, color gradients).  
In deeper layers, they capture **complex patterns** (eyes, wheels, fur textures).

> The network **learns** the filter values during training via backpropagation - we don't hand-craft them.

---

#### 8.4.3.2 FEATURE MAPS - THE OUTPUT OF A FILTER

To apply a filter, we **slide** it across the entire image, one small region at a time (this is the *convolution* operation).

At each position, we compute a **dot product** between the filter weights and the pixel values in that region.  
The result is a single number that measures **how similar** that region is to the pattern the filter represents.

```
Image patch:         Filter:         Dot product:
 [ 10  20  30 ]     [ 1  0 -1 ]     (10×1 + 20×0 + 30×(-1))
 [ 10  20  30 ]  ·  [ 1  0 -1 ]  =   + (10×1 + 20×0 + 30×(-1))
 [ 10  20  30 ]     [ 1  0 -1 ]       + (10×1 + 20×0 + 30×(-1))
                                    =  -60  (strong left > right edge)
```

Doing this for **every position** in the image produces a 2D grid of values called a **feature map** (or *activation map*).

- **High values** > the filter's pattern is present at that location.
- **Low values** > no match.

```
  1 Filter  applied to  1 Image  =  1 Feature Map
  N Filters applied to  1 Image  =  N Feature Maps
             (the output of one conv layer)
```

---

#### 8.4.3.3 STACKING CONVOLUTIONAL LAYERS

The output of one convolutional layer (N feature maps) becomes the **input** to the next convolutional layer.  
The next layer's filters now operate across all N channels simultaneously.

```
Input Image (3 channels: RGB)
    │
    ▼
 Conv Layer 1  >  32 feature maps  (detects: edges, colors)
    │
    ▼
 Conv Layer 2  >  64 feature maps  (detects: corners, textures)
    │
    ▼
 Conv Layer 3  > 128 feature maps  (detects: parts, shapes)
    │
    ▼
    ...
```

Each layer builds on the previous one, combining simpler features into increasingly complex representations. This is called **hierarchical feature learning** and is one of the most powerful properties of deep networks.

| Layer depth | What is typically learned |
|---|---|
| Layer 1 (shallow) | Edges, color gradients |
| Layer 2-3 | Corners, textures, blobs |
| Layer 4-5 | Object parts (eyes, wheels) |
| Layer 6+ (deep) | Whole objects, semantic concepts |

#### 8.4.4 DENSE LAYERS & MAKING PREDICTIONS

After several convolutional layers, the feature maps are **flattened** into a single long 1D vector.  
For example, a 299×299 RGB image processed through several conv layers might be compressed into a vector of size **2,048**.

```
Feature maps  >  Flatten  >  [x₁, x₂, x₃, ..., x₂₀₄₈]  (1D vector)
```

This vector is a **compact, learned representation** of the image - it encodes all the information the network extracted. This step is sometimes called the **embedding** or **bottleneck**.

---

#### 8.4.4.1 BINARY CLASSIFICATION

**Question:** *Is this image a t-shirt or not?*

We connect the feature vector to a **single output neuron** using logistic regression:

```
ŷ = sigmoid(wᵀx + b)
```

Where:
- `x` = feature vector (e.g. 2,048 values)
- `w` = learned weight vector (same size as x)
- `b` = bias term
- `sigmoid(z) = 1 / (1 + e⁻ᶻ)` > squashes output to range [0, 1]

The output `ŷ` is the **probability** that the image belongs to the positive class.

```
ŷ > 0.5  >  "t-shirt"
ŷ ≤ 0.5  >  "not a t-shirt"
```

---

#### 8.4.4.2 MULTI-CLASS CLASSIFICATION

**Question:** *Is this a shirt, t-shirt, or dress?*

We use a separate weight vector `wₖ` for each class `k`, and replace sigmoid with **softmax**:

```
softmax(zₖ) = e^zₖ / Σⱼ e^zⱼ
```

The softmax function guarantees that all output probabilities are:
- between 0 and 1
- sum to exactly 1

```
x (feature vector)
    │
    ├─► w_shirt  > z₁ ─┐
    ├─► w_tshirt > z₂ ─┤─► softmax ─► [P(shirt), P(t-shirt), P(dress)]
    └─► w_dress  > z₃ ─┘                        ↑
                                        sums to 1.0
```

**This collection of logistic regressions, all sharing the same input, is precisely what a Dense layer is.**

---

#### DENSE LAYER AS MATRIX MULTIPLICATION

Stacking all weight vectors `w₁, w₂, ..., wₖ` as rows of a matrix `W`, the entire operation becomes:

```
z = W · x + b
```

This is just **matrix multiplication** - fast, parallelizable, and the backbone of every dense layer.  
The word *"dense"* (or *"fully connected"*) refers to the fact that **every input element connects to every output element** through the weight matrix.

#### 8.4.5 OTHER IMPORTANT LAYERS

#### 8.4.5.1 ACTIVATION FUNCTIONS (ReLU)

After every conv or dense layer, we typically apply an **activation function** to introduce non-linearity.  
Without it, stacking layers would collapse into a single linear transformation.

The most common choice is **ReLU** (Rectified Linear Unit):

```
ReLU(x) = max(0, x)

   Output
    │     /
    │    /
    │   /
    │__/_________ Input
   (negative values > 0)
```

ReLU is fast to compute, reduces the vanishing gradient problem, and works well in practice.

---

#### 8.4.5.2 POOLING LAYERS

A **pooling layer** reduces the spatial size of feature maps - shrinking the data while retaining the most important information.

The most common variant is **Max Pooling**: take a 2×2 region and keep only the **maximum** value.

```
Before Max Pooling (4×4):       After Max Pooling (2×2, stride 2):

  [ 1   3  | 2   4 ]                 [ 6   8 ]
  [ 5   6  | 8   2 ]       >         [ 7   9 ]
  [--------|-------]
  [ 2   1  | 7   9 ]
  [ 3   7  | 4   1 ]
```

**Why pool?**
- Reduces the number of parameters > faster training, less overfitting
- Introduces a degree of **translation invariance** (small shifts in the image don't change the output much)
- Controls the spatial resolution as we go deeper into the network

```
Typical CNN block structure:

  Conv Layer  >  ReLU  >  Pooling  >  (repeat)
```

#### 8.4.6 END-TO-END SUMMARY

Here is the complete picture, from raw pixels to final prediction:

```
┌────────────────────────────────────────────────────────────────┐
│                      Input Image                               │
│                    (e.g. 224×224×3)                            │
└──────────────────────────┬─────────────────────────────────────┘
                           │
              ┌────────────▼────────────┐
              │  Conv Layer + ReLU      │  > detects low-level features
              │  Pooling                │  > reduces spatial size
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  Conv Layer + ReLU      │  > detects mid-level features
              │  Pooling                │
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  Conv Layer + ReLU      │  > detects high-level features
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  Flatten                │  > 1D vector (e.g. 2048 values)
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  Dense Layer + ReLU     │  > learns class-relevant combos
              └────────────┬────────────┘
                           │
              ┌────────────▼────────────┐
              │  Output Layer           │
              │  sigmoid  (binary)      │
              │  softmax  (multi-class) │
              └────────────┬────────────┘
                           │
                    Final Prediction
                 e.g. P(t-shirt) = 0.91
```

#### EACH COMPONENT'S ROLE - AT A GLANCE

| Component | Role |
|---|---|
| **Convolutional Layer** | Extract local visual features using learnable filters |
| **ReLU Activation** | Introduce non-linearity; suppress negative values |
| **Pooling Layer** | Reduce spatial dimensions; improve efficiency & invariance |
| **Flatten** | Convert 3D feature maps into a 1D vector |
| **Dense Layer** | Combine features into class scores (matrix multiplication) |
| **Sigmoid / Softmax** | Convert raw scores into interpretable probabilities |

#### 8.4.8 FURTHER READING

- [CS231n: Convolutional Neural Networks for Visual Recognition](http://cs231n.github.io/) - Stanford's definitive course notes on CNNs
- [3Blue1Brown - But what is a neural network?](https://www.youtube.com/watch?v=aircAruvnKk) - Excellent visual intuition
- [Distill.pub - Feature Visualization](https://distill.pub/2017/feature-visualization/) - Interactive visualization of what filters actually learn

#### TRANSFER LEARNING